## PART A — DATA PREPARATION

### Step 1 — Load & Document Dataset Properties

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load datasets
import pandas
sentiment = pd.read_csv("../data/fear_greed_index.csv")
trades = pd.read_csv("../data/historical_data.csv")

In [4]:
sentiment.columns

Index(['timestamp', 'value', 'classification', 'date'], dtype='object')

In [5]:
trades.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [6]:
sentiment.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [7]:
trades.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [8]:
# Shape
print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)

Sentiment shape: (2644, 4)
Trades shape: (211224, 16)


In [9]:
# Missing values
print("\nMissing Values:")
print(sentiment.isnull().sum())
print(trades.isnull().sum())


Missing Values:
timestamp         0
value             0
classification    0
date              0
dtype: int64
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64


In [10]:
# Duplicates
print("\nDuplicates:")
print("Sentiment:", sentiment.duplicated().sum())
print("Trades:", trades.duplicated().sum())


Duplicates:
Sentiment: 0
Trades: 0


### Step 2 — Convert & Align Dates

In [11]:
# Sentiment date conversion
sentiment["date"] = pd.to_datetime(sentiment["date"])
sentiment["date"] = sentiment["date"].dt.date
print(sentiment.head(2))
sentiment["date"].dtype

    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02


dtype('O')

In [12]:
# Trades timestamp conversion
trades["Timestamp IST"] = pd.to_datetime(
    trades["Timestamp IST"],
    format="%d-%m-%Y %H:%M"
)
trades["date"] = trades["Timestamp IST"].dt.date
trades.head(2)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02


### Merging

In [13]:
merged = trades.merge(
    sentiment[["date", "classification", "value"]],
    on="date",
    how="left"
)

print("Missing sentiment after merge:",
      merged["classification"].isnull().sum())

print("after merging shape",merged.shape)
# as null is there dropping nulls as only 6so we can drop as we have large number of rows
merged=merged.dropna()
print("after dropping nulls shape",merged.shape)


Missing sentiment after merge: 6
after merging shape (211224, 19)
after dropping nulls shape (211218, 19)


### Step 3 — Required Key Metrics

#### Daily PnL per Account

In [14]:
daily_pnl = (
    merged.groupby(["Account", "date"])["Closed PnL"]
    .sum()
    .reset_index()
)
daily_pnl.head()

,Account,date,Closed PnL
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,-21227.0
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,1603.1


#### Win Rate

In [15]:
merged["Win"] = merged["Closed PnL"] > 0

win_rate = (
    merged.groupby("Account")["Win"]
    .mean()
    .reset_index()
)
win_rate.head()

,Account,Win
0,0x083384f897ee0f19899168e3b1bec365f52a9012,0.359612
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,0.442720
2,0x271b280974205ca63b716753467d5a371de622ab,0.301917
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,0.438585
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,0.519914


#### Average Trade Size

In [16]:
avg_trade_size = (
    merged.groupby("Account")["Size USD"]
    .mean()
    .reset_index()
)
avg_trade_size.head()

,Account,Size USD
0,0x083384f897ee0f19899168e3b1bec365f52a9012,16159.576734
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,1653.226327
2,0x271b280974205ca63b716753467d5a371de622ab,8893.000898
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,507.626933
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3138.894782


#### Leverage Distribution

In [17]:
print("The dataset does not include margin or account equity information, therefore true leverage cannot be computed. As a proxy for risk exposure, we use trade size (Size USD) relative to trader activity to approximate leverage behavior.")

The dataset does not include margin or account equity information, therefore true leverage cannot be computed. As a proxy for risk exposure, we use trade size (Size USD) relative to trader activity to approximate leverage behavior.


In [18]:
merged.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'date', 'classification', 'value', 'Win'],
      dtype='object')

##### Proxy 1 — Relative Position Size per Trader

In [19]:
account_avg_size = merged.groupby("Account")["Size USD"].mean()

merged["Size_vs_Avg"] = merged.apply(
    lambda x: x["Size USD"] / account_avg_size[x["Account"]],
    axis=1
)
merged.head(2)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Order ID,Crossed,Fee,Trade ID,Timestamp,date,classification,value,Win,Size_vs_Avg
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,2.642159
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.042854


##### Proxy 2 — Daily Exposure per Trader

In [20]:
daily_exposure = (
    merged.groupby(["Account", "date"])["Size USD"]
    .sum()
    .reset_index()
)
daily_exposure.head(2)

,Account,date,Size USD
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,900880.13
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,542413.18


#### Number of Trades per Day

In [21]:
trades_per_day = (
    merged.groupby(["Account", "date"])
    .size()
    .reset_index(name="Trades_per_day")
)
trades_per_day

,Account,date,Trades_per_day
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,177
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,68
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,40
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,12
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,27
...,...,...,...
2335,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,2025-04-27,82
2336,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,2025-04-28,430
2337,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,2025-04-29,902
2338,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,2025-04-30,75


#### Long / Short Ratio

In [22]:
long_short = (
    merged.groupby(["Account", "Side"])
    .size()
    .unstack(fill_value=0)
)

long_short["Long_ratio"] = (
    long_short.get("Long", 0) /
    long_short.sum(axis=1)
)
long_short.head()

Side,BUY,SELL,Long_ratio
Account,,,
0x083384f897ee0f19899168e3b1bec365f52a9012,1711,2107,0.0
0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,3055,4225,0.0
0x271b280974205ca63b716753467d5a371de622ab,1566,2243,0.0
0x28736f43f1e871e6aa8b1148d38d4994275d72c4,6699,6612,0.0
0x2c229d22b100a7beb69122eed721cee9b24011dd,1179,2060,0.0


# PART B — ANALYSIS

## 1 Does performance differ between Fear vs Greed?

In [23]:
merged.groupby("classification")["Closed PnL"].mean()

classification
Extreme Fear     34.537862
Extreme Greed    67.892861
Fear             54.290400
Greed            42.743559
Neutral          34.307718
Name: Closed PnL, dtype: float64

In [24]:
merged.groupby("classification")["Win"].mean()

classification
Extreme Fear     0.370607
Extreme Greed    0.464943
Fear             0.420768
Greed            0.384828
Neutral          0.396991
Name: Win, dtype: float64

In [25]:
daily_account = (
    merged.groupby(["Account", "date"])
    .agg({
        "Closed PnL": "sum",
        "classification": "first"   # bring sentiment
    })
    .reset_index()
)
daily_account["Cumulative"] = (
    daily_account.groupby("Account")["Closed PnL"]
    .cumsum()
)

daily_account["Rolling_Max"] = (
    daily_account.groupby("Account")["Cumulative"]
    .cummax()
)

daily_account["Drawdown"] = (
    daily_account["Cumulative"] -
    daily_account["Rolling_Max"]
)
daily_account.head(2)

,Account,date,Closed PnL,classification,Cumulative,Rolling_Max,Drawdown
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0,Extreme Greed,0.0,0.0,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0,Extreme Greed,0.0,0.0,0.0


In [26]:
print("We use cumulative PnL deviation from rolling max as drawdown proxy.")

We use cumulative PnL deviation from rolling max as drawdown proxy.


## 2 Do traders change behavior based on sentiment?

In [27]:
merged.groupby("classification").size()

classification
Extreme Fear     21400
Extreme Greed    39992
Fear             61837
Greed            50303
Neutral          37686
dtype: int64

In [28]:
merged.groupby("classification")["Size USD"].mean()

classification
Extreme Fear     5349.731843
Extreme Greed    3112.251565
Fear             7816.109931
Greed            5736.884375
Neutral          4782.732661
Name: Size USD, dtype: float64

In [29]:
merged.groupby(["classification", "Side"]).size()

classification  Side
Extreme Fear    BUY     10935
                SELL    10465
Extreme Greed   BUY     17940
                SELL    22052
Fear            BUY     30270
                SELL    31567
Greed           BUY     24576
                SELL    25727
Neutral         BUY     18969
                SELL    18717
dtype: int64

## 3 Segmentation

### Segment 1 — High vs Low Exposure (Leverage Proxy)

In [30]:
# Create daily exposure with sentiment and pnl
daily_exposure = (
    merged.groupby(["Account", "date"])
    .agg({
        "Size USD": "sum",
        "Closed PnL": "sum",
        "classification": "first"
    })
    .reset_index()
)

# Create exposure segment (proxy for leverage)
daily_exposure["Exposure_Segment"] = pd.qcut(
    daily_exposure["Size USD"],
    2,
    labels=["Low_Exposure", "High_Exposure"]
)

daily_exposure.head()

,Account,date,Size USD,Closed PnL,classification,Exposure_Segment
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,900880.13,0.0,Extreme Greed,High_Exposure
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,542413.18,0.0,Extreme Greed,High_Exposure
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,949380.00,0.0,Extreme Greed,High_Exposure
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,338240.00,-21227.0,Extreme Greed,High_Exposure
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,465700.00,1603.1,Extreme Greed,High_Exposure


### performance across exposure + sentiment

In [31]:
daily_exposure.groupby(
    ["Exposure_Segment", "classification"]
)["Closed PnL"].mean()

/tmp/ipykernel_9669/2221773531.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  daily_exposure.groupby(


Exposure_Segment  classification
Low_Exposure      Extreme Fear       388.267107
                  Extreme Greed     1513.085284
                  Fear               853.865897
                  Greed              695.515964
                  Neutral            183.559781
High_Exposure     Extreme Fear      7667.702713
                  Extreme Greed     9099.205644
                  Fear              9775.447943
                  Greed             6090.547483
                  Neutral           6525.021946
Name: Closed PnL, dtype: float64

### Segment 2 — Frequent vs Infrequent Traders

In [32]:
# Total trades per account
account_freq = (
    merged.groupby("Account")
    .size()
    .reset_index(name="Total_Trades")
)

# Create frequency segment
account_freq["Freq_Segment"] = pd.qcut(
    account_freq["Total_Trades"],
    2,
    labels=["Low_Freq", "High_Freq"]
)

# Merge back
merged = merged.merge(
    account_freq[["Account", "Freq_Segment"]],
    on="Account"
)

merged.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Crossed,Fee,Trade ID,Timestamp,date,classification,value,Win,Size_vs_Avg,Freq_Segment
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,...,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,2.642159,Low_Freq
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,...,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.042854,Low_Freq
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,Buy,0.0,...,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.386190,Low_Freq
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,Buy,0.0,...,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.383307,Low_Freq
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,Buy,0.0,...,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.023410,Low_Freq


In [33]:
merged.groupby(
    ["Freq_Segment", "classification"]
)["Closed PnL"].mean()

/tmp/ipykernel_9669/2039663314.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged.groupby(


Freq_Segment  classification
Low_Freq      Extreme Fear       35.941979
              Extreme Greed     103.264685
              Fear               80.835791
              Greed             210.210287
              Neutral            31.845656
High_Freq     Extreme Fear       34.163353
              Extreme Greed      62.870911
              Fear               51.323784
              Greed              25.004641
              Neutral            34.579701
Name: Closed PnL, dtype: float64

### Segment 3 — Consistent vs Inconsistent Traders

In [34]:
# Total pnl per account
account_total_pnl = (
    merged.groupby("Account")["Closed PnL"]
    .sum()
)

# Top 30% as consistent
threshold = account_total_pnl.quantile(0.7)

consistent_df = (account_total_pnl >= threshold).reset_index()
consistent_df.columns = ["Account", "Consistent"]

# Merge
merged = merged.merge(consistent_df, on="Account")

merged.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Fee,Trade ID,Timestamp,date,classification,value,Win,Size_vs_Avg,Freq_Segment,Consistent
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,...,0.345404,8.950000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,2.642159,Low_Freq,False
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,...,0.005600,4.430000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.042854,Low_Freq,False
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,Buy,0.0,...,0.050431,6.600000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.386190,Low_Freq,False
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,Buy,0.0,...,0.050043,1.080000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.383307,Low_Freq,False
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,Buy,0.0,...,0.003055,1.050000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,False,0.023410,Low_Freq,False


In [36]:
merged.groupby(
    ["Consistent", "classification"]
)["Closed PnL"].mean()

Consistent  classification
False       Extreme Fear       -4.917955
            Extreme Greed      31.211940
            Fear               30.814145
            Greed               1.378552
            Neutral             9.163207
True        Extreme Fear       60.765008
            Extreme Greed     136.960455
            Fear               66.979192
            Greed              93.526109
            Neutral            53.831636
Name: Closed PnL, dtype: float64

## Key Insights

### Based on your numbers earlier, strong insights already visible:

### Insight 1

Extreme Greed days show highest mean PnL and highest win rate.

### Insight 2

Average position size is highest during Fear regimes → traders increase exposure in volatile markets.

### Insight 3

High exposure traders experience significantly larger PnL swings across regimes compared to low exposure traders.



# PART C — Actionable Output

## Strategy 1 — Volatility Risk Control

####     If sentiment ∈ {Fear, Extreme Fear}
####     AND trader ∈ High_Exposure
####     Reduce position size by 20%

## Strategy 2 — Momentum Allocation